In [ ]:
## 1대1 영상 가져오면 초당 몇장씩 이미지로 변환해서 데이터 세트 만들어주는 코드

import cv2
from pathlib import Path
from datetime import datetime

# =========================
# 사용자 설정
# =========================
VIDEO_PATH = "carrot2.mp4"
N = 200                      # 저장할 이미지 장수
INTERVAL_SEC = 0.5          # 0.5초마다 1장
OUTPUT_ROOT = "frames"      # 결과 저장 상위 폴더
IMG_SIZE = 640              # 640x640

# =========================
# 출력 폴더 생성
# =========================
now_str = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = Path(OUTPUT_ROOT) / f"carrot_frames_{now_str}"
output_dir.mkdir(parents=True, exist_ok=True)

# =========================
# 비디오 열기
# =========================
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"비디오를 열 수 없습니다: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration_sec = frame_count / fps if fps > 0 else 0

print(f"[INFO] video: {VIDEO_PATH}")
print(f"[INFO] fps: {fps}")
print(f"[INFO] frame_count: {frame_count}")
print(f"[INFO] duration: {duration_sec:.2f} sec")
print(f"[INFO] output_dir: {output_dir}")

# =========================
# 0.5초마다 프레임 저장
# =========================
saved_count = 106

for i in range(N):
    target_time_sec = i * INTERVAL_SEC

    if target_time_sec > duration_sec:
        print("[INFO] 비디오 길이를 초과해서 중단합니다.")
        break

    # 해당 시간 위치로 이동
    cap.set(cv2.CAP_PROP_POS_MSEC, target_time_sec * 1000)

    ret, frame = cap.read()
    if not ret:
        print(f"[WARN] {target_time_sec:.2f}s 프레임을 읽지 못했습니다.")
        break

    # 640x640 리사이즈
    frame_resized = cv2.resize(frame, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

    # 저장
    save_path = output_dir / f"frame_{saved_count:04d}_{target_time_sec:.1f}s.jpg"
    cv2.imwrite(str(save_path), frame_resized)

    saved_count += 1

cap.release()

print(f"[DONE] 저장 완료: {saved_count}장")
print(f"[DONE] 저장 위치: {output_dir}")

[INFO] video: carrot2.mp4
[INFO] fps: 24.0
[INFO] frame_count: 1609
[INFO] duration: 67.04 sec
[INFO] output_dir: frames\carrot_frames_20260609_223732
[INFO] 비디오 길이를 초과해서 중단합니다.
[DONE] 저장 완료: 241장
[DONE] 저장 위치: frames\carrot_frames_20260609_223732


In [ ]:
## 폴더내 이미지를 클래스 별로 0부터 카운트해서 이미지 재정렬 해주는 코드

from pathlib import Path

# =========================
# 사용자 설정
# =========================
TARGET_DIR = "frames/carrot_frames_20260609_222913"  # 이미지가 들어있는 폴더
NEW_PREFIX = "big_carrot"

IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

# =========================
# 이미지 파일 목록 가져오기
# =========================
target_dir = Path(TARGET_DIR)

if not target_dir.exists():
    raise FileNotFoundError(f"폴더가 없습니다: {target_dir}")

image_files = sorted([
    p for p in target_dir.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
])

print(f"[INFO] 이미지 개수: {len(image_files)}")

# =========================
# 이름 충돌 방지를 위해 임시 이름으로 먼저 변경
# =========================
temp_files = []

for i, img_path in enumerate(image_files):
    temp_path = target_dir / f"__temp_rename_{i:04d}{img_path.suffix.lower()}"
    img_path.rename(temp_path)
    temp_files.append(temp_path)

# =========================
# 최종 이름으로 변경
# =========================
for i, temp_path in enumerate(temp_files):
    new_name = f"{NEW_PREFIX}_{i:04d}{temp_path.suffix.lower()}"
    new_path = target_dir / new_name

    temp_path.rename(new_path)
    print(f"{temp_path.name} -> {new_path.name}")

print("[DONE] 파일명 변경 완료")

[INFO] 이미지 개수: 240
__temp_rename_0000.jpg -> big_carrot_0000.jpg
__temp_rename_0001.jpg -> big_carrot_0001.jpg
__temp_rename_0002.jpg -> big_carrot_0002.jpg
__temp_rename_0003.jpg -> big_carrot_0003.jpg
__temp_rename_0004.jpg -> big_carrot_0004.jpg
__temp_rename_0005.jpg -> big_carrot_0005.jpg
__temp_rename_0006.jpg -> big_carrot_0006.jpg
__temp_rename_0007.jpg -> big_carrot_0007.jpg
__temp_rename_0008.jpg -> big_carrot_0008.jpg
__temp_rename_0009.jpg -> big_carrot_0009.jpg
__temp_rename_0010.jpg -> big_carrot_0010.jpg
__temp_rename_0011.jpg -> big_carrot_0011.jpg
__temp_rename_0012.jpg -> big_carrot_0012.jpg
__temp_rename_0013.jpg -> big_carrot_0013.jpg
__temp_rename_0014.jpg -> big_carrot_0014.jpg
__temp_rename_0015.jpg -> big_carrot_0015.jpg
__temp_rename_0016.jpg -> big_carrot_0016.jpg
__temp_rename_0017.jpg -> big_carrot_0017.jpg
__temp_rename_0018.jpg -> big_carrot_0018.jpg
__temp_rename_0019.jpg -> big_carrot_0019.jpg
__temp_rename_0020.jpg -> big_carrot_0020.jpg
__temp_rename_0

In [ ]:
# 폴더내 txt파일내 ID가 0으로 통일되어 있다면 이 코드를 이용해서 ID 수정 가능

from pathlib import Path

# =========================
# 사용자 설정
# =========================
LABEL_DIR = "big_carrot/aa"   # txt 파일들이 들어있는 폴더 경로
NEW_CLASS_ID = 7       # 바꾸고 싶은 클래스 번호

# =========================
# 라벨 파일 처리
# =========================
label_dir = Path(LABEL_DIR)

if not label_dir.exists():
    raise FileNotFoundError(f"폴더가 없습니다: {label_dir}")

txt_files = sorted(label_dir.glob("*.txt"))

print(f"[INFO] txt 파일 개수: {len(txt_files)}")
print(f"[INFO] 변경할 class id: {NEW_CLASS_ID}")

for txt_path in txt_files:
    new_lines = []

    with open(txt_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()

        # 빈 줄은 유지
        if not line:
            new_lines.append("")
            continue

        parts = line.split()

        # YOLO 라벨은 보통 5개: class x y w h
        if len(parts) < 5:
            print(f"[WARN] 형식 이상, 건너뜀: {txt_path.name} -> {line}")
            new_lines.append(line)
            continue

        # 맨 앞 class id만 변경
        parts[0] = str(NEW_CLASS_ID)

        new_lines.append(" ".join(parts))

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(new_lines) + "\n")

    print(f"[OK] {txt_path.name}")

print("[DONE] 모든 txt 라벨 class id 변경 완료")

[INFO] txt 파일 개수: 227
[INFO] 변경할 class id: 3
[OK] big_carrot_0000.txt
[OK] big_carrot_0001.txt
[OK] big_carrot_0002.txt
[OK] big_carrot_0003.txt
[OK] big_carrot_0004.txt
[OK] big_carrot_0005.txt
[OK] big_carrot_0006.txt
[OK] big_carrot_0007.txt
[OK] big_carrot_0008.txt
[OK] big_carrot_0009.txt
[OK] big_carrot_0010.txt
[OK] big_carrot_0011.txt
[OK] big_carrot_0012.txt
[OK] big_carrot_0013.txt
[OK] big_carrot_0014.txt
[OK] big_carrot_0015.txt
[OK] big_carrot_0016.txt
[OK] big_carrot_0017.txt
[OK] big_carrot_0018.txt
[OK] big_carrot_0019.txt
[OK] big_carrot_0020.txt
[OK] big_carrot_0021.txt
[OK] big_carrot_0022.txt
[OK] big_carrot_0023.txt
[OK] big_carrot_0024.txt
[OK] big_carrot_0025.txt
[OK] big_carrot_0026.txt
[OK] big_carrot_0027.txt
[OK] big_carrot_0028.txt
[OK] big_carrot_0029.txt
[OK] big_carrot_0030.txt
[OK] big_carrot_0031.txt
[OK] big_carrot_0032.txt
[OK] big_carrot_0033.txt
[OK] big_carrot_0034.txt
[OK] big_carrot_0035.txt
[OK] big_carrot_0036.txt
[OK] big_carrot_0037.txt
[OK] 